In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
from kaggle_secrets import UserSecretsClient
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key=UserSecretsClient().get_secret("ROBOFLOW_API_KEY"))
project = rf.workspace("aa-qgksi").project("road-damage-ehnbm")
version = project.version(1)
dataset = version.download("yolov8")
                

In [ ]:
!pip install ultralytics

In [ ]:
import os
import json
import time
import random
import numpy              as     np
import pandas             as     pd
import matplotlib.pyplot  as     plt
import matplotlib.patches as patches
from PIL                import Image
from roboflow           import Roboflow
from datetime           import datetime
from ultralytics        import YOLO

In [ ]:
print("completed successfully", dataset.location)

In [ ]:
import os

DATASET_DIR = "/kaggle/working/road-damage-1"

for root, dirs, files in os.walk(DATASET_DIR):
    level = root.replace(DATASET_DIR, "").count(os.sep)

    if level > 3:
        dirs[:] = []
        continue

    print("  " * level + os.path.basename(root) + "/")

    for file in files[:5]:
        print("  " * (level + 1) + file)

In [ ]:
from pathlib import Path
import yaml
import os

# ==============================
# Dataset paths
# ==============================
DATASET_DIR = "/kaggle/working/road-damage-1"

DATA_YAML = f"{DATASET_DIR}/data.yaml"

TRAIN_IMAGES_DIR = f"{DATASET_DIR}/train/images"
TRAIN_LABELS_DIR = f"{DATASET_DIR}/train/labels"

VAL_IMAGES_DIR = f"{DATASET_DIR}/valid/images"
VAL_LABELS_DIR = f"{DATASET_DIR}/valid/labels"

TEST_IMAGES_DIR = f"{DATASET_DIR}/test/images"
TEST_LABELS_DIR = f"{DATASET_DIR}/test/labels"

# ==============================
# Experiment outputs
# ==============================
EXPERIMENTS_DIR = "/kaggle/working/experiments_road_damage"

OUTPUTS_DIR = "/kaggle/working/outputs_road_damage"
RESULTS_JSON = f"{OUTPUTS_DIR}/results.json"

PLOTS_DIR = f"{OUTPUTS_DIR}/plots"
PREDICTIONS_DIR = f"{OUTPUTS_DIR}/predictions"

# ==============================
# Evaluation settings
# ==============================
IOU_THRESHOLD = 0.5
NUM_IMAGES_TO_SHOW = 9

# ==============================
# Create directories
# ==============================
for folder in [
    EXPERIMENTS_DIR,
    OUTPUTS_DIR,
    PLOTS_DIR,
    PREDICTIONS_DIR
]:
    os.makedirs(folder, exist_ok=True)

# ==============================
# Check dataset
# ==============================
assert os.path.exists(DATA_YAML), "data.yaml not found!"

for folder in [
    TRAIN_IMAGES_DIR,
    TRAIN_LABELS_DIR,
    VAL_IMAGES_DIR,
    VAL_LABELS_DIR,
    TEST_IMAGES_DIR,
    TEST_LABELS_DIR
]:
    assert os.path.isdir(folder), f"Missing folder: {folder}"

print("Dataset paths verified successfully!")
print("Dataset:", DATASET_DIR)
print("YAML:", DATA_YAML)

In [ ]:
import yaml

with open(DATA_YAML, "r") as file:
    data_config = yaml.safe_load(file)

print("Number of classes:", data_config.get("nc"))
print("Class names:", data_config.get("names"))
print("Train:", data_config.get("train"))
print("Validation:", data_config.get("val"))
print("Test:", data_config.get("test"))

In [ ]:
MODELS = ["yolov8n.pt", "yolov8s.pt", "yolov8m.pt"]

In [ ]:
EPOCHS = 30
BATCH_SIZE = 16
IMAGE_SIZE = 640
LEARNING_RATE = 0.01
OPTIMIZER = "auto"      
SEED = 42

In [ ]:

import os
import time
from datetime import datetime

from ultralytics import YOLO


def train_one_model(weights_name: str) -> dict:
    model_name = weights_name.replace(".pt", "")
    run_name = f"{model_name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

    print(f"\n=== Start training {model_name} ===")

    # 1. Train
    model = YOLO(weights_name)

    model.train(
        data=DATA_YAML,
        epochs=EPOCHS,
        batch=BATCH_SIZE,
        imgsz=IMAGE_SIZE,
        lr0=LEARNING_RATE,
        optimizer=OPTIMIZER,
        seed=SEED,
        project=EXPERIMENTS_DIR,
        name=run_name,
    )

    # 2. Load best weights
    best_weights_path = os.path.join(
        EXPERIMENTS_DIR, run_name, "weights", "best.pt"
    )

    if not os.path.exists(best_weights_path):
        raise FileNotFoundError(
            f"Best weights not found: {best_weights_path}"
        )

    best_model = YOLO(best_weights_path)

    # 3. Validate on validation split
    metrics = best_model.val(
        data=DATA_YAML,
        split="val",
        imgsz=IMAGE_SIZE,
        batch=BATCH_SIZE,
    )

    map50 = metrics.box.map50
    map50_95 = metrics.box.map
    precision = metrics.box.mp
    recall = metrics.box.mr

    # 4. Model size
    model_size_mb = os.path.getsize(best_weights_path) / (1024 * 1024)

    # 5. Inference speed using a validation image
    image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

    sample_images = sorted([
        f for f in os.listdir(VAL_IMAGES_DIR)
        if f.lower().endswith(image_extensions)
    ]) if os.path.isdir(VAL_IMAGES_DIR) else []

    fps = None
    avg_inference_ms = None

    if sample_images:
        sample_path = os.path.join(VAL_IMAGES_DIR, sample_images[0])

        # Warm-up
        best_model.predict(
            source=sample_path,
            imgsz=IMAGE_SIZE,
            verbose=False
        )

        n_runs = 20
        start = time.perf_counter()

        for _ in range(n_runs):
            best_model.predict(
                source=sample_path,
                imgsz=IMAGE_SIZE,
                verbose=False
            )

        elapsed = time.perf_counter() - start

        avg_inference_ms = (elapsed / n_runs) * 1000
        fps = 1000 / avg_inference_ms if avg_inference_ms > 0 else None

    # 6. Collect results
    result = {
        "model_name": model_name,
        "best_weights_path": best_weights_path,
        "mAP50": round(float(map50), 4),
        "mAP50-95": round(float(map50_95), 4),
        "precision": round(float(precision), 4),
        "recall": round(float(recall), 4),
        "model_size_mb": round(model_size_mb, 2),
        "avg_inference_ms": (
            round(avg_inference_ms, 2)
            if avg_inference_ms is not None else None
        ),
        "fps": round(fps, 2) if fps is not None else None,
    }

    print(f"Model: {model_name}")
    print(f"Best weights: {best_weights_path}")
    print(
        f"mAP50={result['mAP50']} | "
        f"mAP50-95={result['mAP50-95']} | "
        f"Precision={result['precision']} | "
        f"Recall={result['recall']} | "
        f"Size={result['model_size_mb']} MB | "
        f"FPS={result['fps']}"
    )

    return result


In [ ]:
def train_all_models():
    results = []
    for weights_name in MODELS:
        result = train_one_model(weights_name)
        results.append(result)
    return results

In [ ]:
def print_comparison_table(results: list):
    print("\n=== Comparison Table ===")
    header = f"{'Model':<10}{'mAP50':<10}{'mAP50-95':<12}{'Precision':<12}{'Recall':<10}{'Size(MB)':<10}{'FPS':<10}"
    print(header)
    print("-" * len(header))
    for r in results:
        print(
            f"{r['model_name']:<10}{r['mAP50']:<10}{r['mAP50-95']:<12}"
            f"{r['precision']:<12}{r['recall']:<10}{r['model_size_mb']:<10}{r['fps']:<10}"
        )
    best = max(results, key=lambda r: r["mAP50-95"])
    print(f"\nBest model for deployment (highest mAP50-95): {best['model_name']}")
    print(f"Weights path: {best['best_weights_path']}")

In [ ]:
def save_results(results: list):
    os.makedirs(os.path.dirname(RESULTS_JSON), exist_ok=True)
    with open(RESULTS_JSON, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(f"\nsaved result: {RESULTS_JSON}")

In [ ]:
all_results = train_all_models()
print_comparison_table(all_results)
save_results(all_results)

In [ ]:
def load_results() -> list:
    if not os.path.exists(RESULTS_JSON):
        raise FileNotFoundError(
            f"No Found File '{RESULTS_JSON}'. "
        )
    with open(RESULTS_JSON, "r", encoding="utf-8") as f:
        return json.load(f)

In [ ]:
def print_comparison_table(results: list):
    print("\n=== Comparison Table ===")
    header = f"{'Model':<10}{'mAP50':<10}{'mAP50-95':<12}{'Precision':<12}{'Recall':<10}{'Size(MB)':<10}{'FPS':<10}"
    print(header)
    print("-" * len(header))
    for r in results:
        print(
            f"{r['model_name']:<10}{r['mAP50']:<10}{r['mAP50-95']:<12}"
            f"{r['precision']:<12}{r['recall']:<10}{r['model_size_mb']:<10}{r['fps']:<10}"
        )
    best = max(results, key=lambda r: r["mAP50-95"])
    print(f"\nBest model (highest mAP50-95): {best['model_name']}")


In [ ]:
def _bar_plot(results: list, metric_key: str, title: str, ylabel: str, filename: str):
    names = [r["model_name"] for r in results]
    values = [r[metric_key] for r in results]
 
    plt.figure(figsize=(6, 4))
    bars = plt.bar(names, values, color=["#4C72B0", "#55A868", "#C44E52"])
    plt.title(title)
    plt.ylabel(ylabel)
    plt.xlabel("Model")
    for bar, value in zip(bars, values):
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            str(value),
            ha="center",
            va="bottom",
        )
 
    plt.tight_layout()
    os.makedirs(PLOTS_DIR, exist_ok=True)
    save_path = os.path.join(PLOTS_DIR, filename)
    plt.savefig(save_path)
    plt.close()
    print(f"Saved Plots: {save_path}")

In [ ]:
def plot_all_comparisons(results: list):
    _bar_plot(results, "mAP50", "mAP50 Comparison", "mAP50", "map50.png")
    _bar_plot(results, "mAP50-95", "mAP50-95 Comparison", "mAP50-95", "map50_95.png")
    _bar_plot(results, "precision", "Precision Comparison", "Precision", "precision.png")
    _bar_plot(results, "recall", "Recall Comparison", "Recall", "recall.png")
    _bar_plot(results, "model_size_mb", "Model Size Comparison", "Size (MB)", "model_size.png")
    _bar_plot(results, "fps", "Inference Speed (FPS) Comparison", "FPS", "fps.png")
 

In [ ]:
all_results = load_results()
print_comparison_table(all_results)
plot_all_comparisons(all_results) 


In [ ]:
import os, json, time, random, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO

with open(RESULTS_JSON) as f:
    results = json.load(f)

for r in results:
    print(r["model_name"], "| weights found:", os.path.exists(r["best_weights_path"]))

test_images = sorted(glob.glob(f"{TEST_IMAGES_DIR}/*.jpg"))
print("Test images:", len(test_images))

In [ ]:
# 0 = Retak_Kulit_Buaya (alligator crack), 1 = Retak_Memanjang (longitudinal crack)
test_rows = []

for r in results:
    name = r["model_name"]
    model = YOLO(r["best_weights_path"])

    m = model.val(
        data=DATA_YAML,
        split="test",
        imgsz=IMAGE_SIZE,
        batch=BATCH_SIZE,
        plots=True,
        project=OUTPUTS_DIR,
        name=f"test_{name}",
        exist_ok=True,
        verbose=False,
    )

    row = {
        "model": name,
        "test_mAP50": m.box.map50,
        "test_mAP50-95": m.box.map,
        "test_precision": m.box.mp,
        "test_recall": m.box.mr,
        "AP50_alligator": m.box.ap50[0],
        "AP50_longitudinal": m.box.ap50[1],
    }
    test_rows.append(row)
    print(name, "| mAP50 =", round(row["test_mAP50"], 3), "| mAP50-95 =", round(row["test_mAP50-95"], 3))

test_df = pd.DataFrame(test_rows)
test_df.round(3)

In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()   # نظّف الـ GPU من الخلية اللي قبلها


def box_iou(a, b):
    # a, b = [x1, y1, x2, y2]
    x1 = max(a[0], b[0])
    y1 = max(a[1], b[1])
    x2 = min(a[2], b[2])
    y2 = min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area_a = (a[2] - a[0]) * (a[3] - a[1])
    area_b = (b[2] - b[0]) * (b[3] - b[1])
    return inter / (area_a + area_b - inter + 1e-9)


def read_gt_boxes(label_path, w, h):
    boxes = []
    if not os.path.exists(label_path):
        return boxes
    for line in open(label_path).read().splitlines():
        if not line.strip():
            continue
        cls, xc, yc, bw, bh = line.split()[:5]
        xc, yc, bw, bh = float(xc) * w, float(yc) * h, float(bw) * w, float(bh) * h
        boxes.append((int(cls), [xc - bw / 2, yc - bh / 2, xc + bw / 2, yc + bh / 2]))
    return boxes


def mean_iou(weights_path, image_paths, conf=0.25, chunk_size=16):
    model = YOLO(weights_path)
    best_ious = []   # أحسن IoU لكل صندوق حقيقي

    # بنشتغل على دفعات صغيرة (16 صورة) عشان الـ GPU متتملاش
    for start in range(0, len(image_paths), chunk_size):
        chunk = image_paths[start:start + chunk_size]
        outputs = model.predict(chunk, imgsz=IMAGE_SIZE, conf=conf, verbose=False)

        for result in outputs:
            h, w = result.orig_shape
            label_path = result.path.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
            gt_boxes = read_gt_boxes(label_path, w, h)

            pred_boxes = result.boxes.xyxy.cpu().numpy()
            pred_cls = result.boxes.cls.cpu().numpy().astype(int)

            for gt_cls, gt_box in gt_boxes:
                best = 0.0
                for pb, pc in zip(pred_boxes, pred_cls):
                    if pc == gt_cls:
                        best = max(best, box_iou(gt_box, pb))
                best_ious.append(best)

    del model
    gc.collect()
    torch.cuda.empty_cache()

    best_ious = np.array(best_ious)
    matched = best_ious[best_ious >= IOU_THRESHOLD]
    return {
        "iou_matched": matched.mean() if len(matched) else 0.0,
        "iou_all": best_ious.mean(),
        "found_%": 100 * len(matched) / len(best_ious),
    }


iou_rows = []
for r in results:
    out = mean_iou(r["best_weights_path"], test_images)
    out["model"] = r["model_name"]
    iou_rows.append(out)
    print(r["model_name"], {k: round(float(v), 3) for k, v in out.items() if k != "model"})

iou_df = pd.DataFrame(iou_rows)
iou_df.round(3)

In [ ]:
from ultralytics.utils.torch_utils import get_flops


def measure_fps(weights_path, n_images=100):
    model = YOLO(weights_path)
    paths = test_images[:n_images]
    model.predict(paths[:10], imgsz=IMAGE_SIZE, verbose=False)   # warm-up
    start = time.perf_counter()
    for p in paths:
        model.predict(p, imgsz=IMAGE_SIZE, verbose=False)
    seconds = time.perf_counter() - start
    return len(paths) / seconds


speed_rows = []
for r in results:
    path = r["best_weights_path"]
    model = YOLO(path)
    params = sum(p.numel() for p in model.model.parameters()) / 1e6
    try:
        gflops = get_flops(model.model, imgsz=IMAGE_SIZE)
    except Exception as e:
        gflops = None
        print("GFLOPs not available:", e)

    row = {
        "model": r["model_name"],
        "fps_pytorch": measure_fps(path),
        "size_mb": os.path.getsize(path) / (1024 * 1024),
        "params_m": params,
        "gflops": gflops,
    }
    speed_rows.append(row)
    print(row["model"], "| FPS =", round(row["fps_pytorch"], 1), "| size =", round(row["size_mb"], 1), "MB")

speed_df = pd.DataFrame(speed_rows)
speed_df.round(2)

In [ ]:
import torch
print(torch.cuda.get_device_name(0))

In [ ]:
onnx_rows = []

for r in results:
    name = r["model_name"]
    onnx_path = YOLO(r["best_weights_path"]).export(format="onnx", imgsz=IMAGE_SIZE)

    row = {
        "model": name,
        "onnx_size_mb": os.path.getsize(onnx_path) / (1024 * 1024),
        "fps_onnx": measure_fps(onnx_path),
    }
    onnx_rows.append(row)
    print(name, "| ONNX FPS =", round(row["fps_onnx"], 1), "| ONNX size =", round(row["onnx_size_mb"], 1), "MB")

onnx_df = pd.DataFrame(onnx_rows)
onnx_df.round(2)

In [ ]:
final_df = (
    test_df
    .merge(iou_df, on="model")
    .merge(speed_df, on="model")
    .merge(onnx_df, on="model")
)

final_path = f"{OUTPUTS_DIR}/final_comparison.csv"
final_df.round(3).to_csv(final_path, index=False)
print("Saved:", final_path)
final_df.round(3)

In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()

TUNE_EPOCHS = 10
TUNE_FRACTION = 0.2   # 20% من الصور بس عشان نوفّر وقت

TUNE_CONFIGS = {
    "base":    {"optimizer": "AdamW", "lr0": 0.001},
    "lr_low":  {"optimizer": "AdamW", "lr0": 0.0003},
    "lr_high": {"optimizer": "AdamW", "lr0": 0.003},
    "aug":     {"optimizer": "AdamW", "lr0": 0.001, "mixup": 0.1, "scale": 0.7, "degrees": 5.0},
}

tune_rows = []

for cfg_name, cfg in TUNE_CONFIGS.items():
    print("\n=== Tuning config:", cfg_name, cfg)
    run_name = f"tune_{cfg_name}"

    model = YOLO("yolov8n.pt")
    model.train(
        data=DATA_YAML,
        epochs=TUNE_EPOCHS,
        fraction=TUNE_FRACTION,
        batch=BATCH_SIZE,
        imgsz=IMAGE_SIZE,
        seed=SEED,
        project=EXPERIMENTS_DIR,
        name=run_name,
        exist_ok=True,
        **cfg,
    )

    best_path = f"{EXPERIMENTS_DIR}/{run_name}/weights/best.pt"
    m = YOLO(best_path).val(data=DATA_YAML, split="val", imgsz=IMAGE_SIZE, batch=BATCH_SIZE, verbose=False)

    tune_rows.append({
        "config": cfg_name,
        "val_mAP50": m.box.map50,
        "val_mAP50-95": m.box.map,
        "val_precision": m.box.mp,
        "val_recall": m.box.mr,
    })
    print(cfg_name, "| mAP50 =", round(m.box.map50, 3), "| mAP50-95 =", round(m.box.map, 3))

tune_df = pd.DataFrame(tune_rows)
tune_df.to_csv(f"{OUTPUTS_DIR}/tuning_results.csv", index=False)
tune_df.round(3)

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/working/experiments_road_damage"):
    if "best.pt" in files and "lr_low" in root.lower():
        print(os.path.join(root, "best.pt"))

In [ ]:
import os
import shutil
from IPython.display import FileLink, display

BEST_MODEL_PATH = "/kaggle/working/experiments_road_damage/tune_lr_low/weights/best.pt"
FINAL_MODEL_PATH = "/kaggle/working/road_damage_best.pt"

shutil.copy2(BEST_MODEL_PATH, FINAL_MODEL_PATH)

print("Model saved successfully!")
print("Size:", round(os.path.getsize(FINAL_MODEL_PATH) / (1024 * 1024), 2), "MB")

display(FileLink(FINAL_MODEL_PATH))